Event-based directional prediction on SPY with CUSUM + Triple-Barrier

This notebook fetches SPY OHLCV, detects events via a CUSUM filter, labels with the triple barrier method, builds a compact feature set, trains a baseline classifier with time-aware splits, and evaluates results. It concludes by producing asset_weights and asset_returns and calling a user-defined backtest exactly once.

Notes
- Data: findata.get_equity_prices
- Event detection and labeling: mlfinlab
- Model: scikit-learn with PurgedKFold from mlfinlab
- Targets: directional labels from triple-barrier

In [1]:
# Imports
import pandas as pd
import numpy as np
from datetime import timedelta

# Data
from findata.equity_prices import get_equity_prices  # fetch SPY OHLCV (see findata docs)

# mlfinlab components
from mlfinlab.filters.filters import cusum_filter  # CUSUM events
from mlfinlab.util.volatility import get_daily_vol  # daily volatility estimator
from mlfinlab.labeling.labeling import add_vertical_barrier, get_events, get_bins  # triple barrier + labels
from mlfinlab.cross_validation.cross_validation import PurgedKFold  # time-aware CV

# Modeling
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix

In [2]:
# 1) Fetch SPY OHLCV daily data
start_date = "2015-01-01"
end_date = "2025-01-01"

ohlcv_df = get_equity_prices(
    tickers=["SPY"], start_date=start_date, end_date=end_date, fields=None, frequency="1d", auto_adjust=True
)
assert set(["Open","High","Low","Close","Volume"]).issubset(ohlcv_df.columns)
print(ohlcv_df.head())
print(ohlcv_df.tail())
print("Shape:", ohlcv_df.shape)

Price            Close        High         Low        Open     Volume
Date                                                                 
2015-01-02  170.125015  171.325830  169.089839  170.911759  121465900
2015-01-05  167.052582  169.247150  166.746174  169.081524  169632600
2015-01-06  165.479126  167.880730  164.684105  167.358997  209151400
2015-01-07  167.541199  167.880740  166.356963  166.804154  125346700
2015-01-08  170.514191  170.729515  168.932451  168.949005  147217800
Price            Close        High         Low        Open    Volume
Date                                                                
2024-12-24  592.702087  592.741554  586.955433  587.537023  33160100
2024-12-26  592.741577  593.865231  589.528182  590.927860  41219100
2024-12-27  586.502014  589.232425  582.312785  588.995807  64969300
2024-12-30  579.809082  583.278769  576.053563  579.483844  56578800
2024-12-31  577.699768  582.194563  576.063470  581.474960  57052700
Shape: (2516, 5)


In [3]:
# 2) Daily volatility for barrier sizing (mlfinlab.util.volatility.get_daily_vol)
close = ohlcv_df["Close"].dropna()
daily_vol = get_daily_vol(close=close, lookback=100).dropna()

# Inspect
display(daily_vol.tail())
print("daily_vol.describe():\n", daily_vol.describe())

Date
2024-12-24    0.011936
2024-12-26    0.011818
2024-12-27    0.011814
2024-12-30    0.011825
2024-12-31    0.011913
Name: Close, dtype: float64

daily_vol.describe():
 count    2514.000000
mean        0.012875
std         0.005628
min         0.004672
25%         0.009356
50%         0.011543
75%         0.015071
max         0.039418
Name: Close, dtype: float64


In [4]:
# 3) Event detection via symmetric CUSUM (mlfinlab.filters.filters.cusum_filter)
# Use a threshold as a fraction of recent volatility; here, a fixed multiple of the median vol.
threshold = float(daily_vol.median() * 0.5)

t_events = cusum_filter(raw_time_series=close, threshold=threshold, time_stamps=True)
print(f"Detected {len(t_events)} events. Sample:")
print(pd.DatetimeIndex(t_events)[:10])

Detected 2509 events. Sample:
DatetimeIndex(['2015-01-05', '2015-01-06', '2015-01-07', '2015-01-08',
               '2015-01-09', '2015-01-12', '2015-01-13', '2015-01-14',
               '2015-01-15', '2015-01-16'],
              dtype='datetime64[ns]', freq=None)


In [5]:
# 4) Vertical barriers (mlfinlab.labeling.labeling.add_vertical_barrier)
# Set a 5-day vertical barrier horizon
vertical_horizon_days = 5

t1 = add_vertical_barrier(
    t_events=pd.Series(index=pd.DatetimeIndex(t_events), dtype="datetime64[ns]"),
    close=close,
    num_days=vertical_horizon_days,
)
# Ensure valid datetimes; replace missing with last available close timestamp to avoid NaN slices
t1 = pd.to_datetime(t1, errors="coerce")
t1 = t1.fillna(close.index[-1])

print("t1 (vertical barriers) sample:")
print(t1.dropna().head())

t1 (vertical barriers) sample:
Series([], dtype: datetime64[ns])


In [6]:
# 5) Triple-barrier events (mlfinlab.labeling.labeling.get_events)
# Profit taking / Stop loss as 1.0 x volatility, require a minimum target return relative to vol
pt_sl = [1.0, 1.0]
min_ret = float(daily_vol.median() * 0.5)

# Align target volatility to event times
target = daily_vol.reindex(close.index).fillna(method='ffill')

events = get_events(
    close=close,
    t_events=pd.DatetimeIndex(t_events),
    pt_sl=pt_sl,
    target=target,
    min_ret=min_ret,
    num_threads=1,
    vertical_barrier_times=t1,
    side_prediction=None,
    verbose=True,
)
print("events columns:", events.columns)
print(events.dropna().head())

/var/folders/jr/prby656n267_0pztcwybgy5r0000gn/T/ipykernel_64174/2919996425.py:7: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  target = daily_vol.reindex(close.index).fillna(method='ffill')


events columns: Index(['t1', 'trgt', 'pt', 'sl'], dtype='object')
                   t1      trgt   pt   sl
2015-01-06 2015-01-07  0.006540  1.0  1.0
2015-01-07 2015-01-08  0.015581  1.0  1.0
2015-01-08 2015-01-15  0.025762  1.0  1.0
2015-01-09 2015-01-15  0.022873  1.0  1.0
2015-01-12 2015-02-11  0.020612  1.0  1.0


In [7]:
# 6) Labels from triple-barrier events (mlfinlab.labeling.labeling.get_bins)

labeled = get_bins(triple_barrier_events=events, close=close)
# Keep only events with labels and targets
labeled = labeled.dropna(subset=["bin"]).copy()
print(labeled.head())
print(labeled["bin"].value_counts(dropna=False))

                 ret  bin
2015-01-06  0.012461  1.0
2015-01-07  0.017745  1.0
2015-01-08 -0.033414 -1.0
2015-01-09 -0.025606 -1.0
2015-01-12  0.021120  1.0
bin
 1.0    1423
-1.0     980
Name: count, dtype: int64


In [8]:
# 7) Feature engineering: compact, sensible set at event times
# We implement a helper that consumes ohlcv_df and labeled, and returns X, y, t1, event_index

def build_event_features_spy(ohlcv_df: pd.DataFrame, labeled: pd.DataFrame,
                              rsi_window: int = 14,
                              mom_windows=(5, 10, 20),
                              vol_window: int = 20):
    px = ohlcv_df["Close"].copy().dropna()
    vol = ohlcv_df["Volume"].astype(float).reindex(px.index).fillna(0.0)
    rets = px.pct_change().fillna(0.0)

    feats = pd.DataFrame(index=px.index)
    # Lagged returns
    feats["ret_1"] = rets.shift(1)
    feats["ret_3"] = px.pct_change(3).shift(1)
    feats["ret_5"] = px.pct_change(5).shift(1)
    # Rolling volatility
    feats["vol_roll"] = rets.rolling(vol_window).std().shift(1)
    # Momentum windows
    for w in mom_windows:
        feats[f"mom_{w}"] = (px / px.shift(w) - 1.0).shift(1)
    # RSI-like oscillator
    delta = px.diff()
    up = (delta.clip(lower=0)).rolling(rsi_window).mean()
    down = (-delta.clip(upper=0)).rolling(rsi_window).mean()
    rs = up / (down.replace(0, np.nan))
    rsi = 100 - 100 / (1 + rs)
    feats["rsi"] = rsi.shift(1) / 100.0  # scale to 0-1
    # Liquidity proxy: volume z-score
    vol_z = (vol - vol.rolling(60).mean()) / (vol.rolling(60).std())
    feats["volz"] = vol_z.shift(1)

    feats = feats.replace([np.inf, -np.inf], np.nan).dropna()

    # Align to labeled events
    events_idx = labeled.index.intersection(feats.index)
    X = feats.loc[events_idx].copy()
    y = labeled.loc[events_idx, "bin"].astype(int)
    # Pull t1 from events DataFrame available in scope
    t1_series = events.loc[events_idx, "t1"]
    return X, y, t1_series, events_idx

# Build features
X, y, t1_used, event_index = build_event_features_spy(ohlcv_df, labeled,
                                                     rsi_window=14, mom_windows=(5,10,20), vol_window=20)
print(X.head())
print("X shape:", X.shape, " | y value counts:\n", y.value_counts())

               ret_1     ret_3     ret_5  vol_roll     mom_5    mom_10  \
2015-03-31  0.012200  0.012102 -0.008333  0.009036 -0.008333  0.002875   
2015-04-01 -0.008740  0.005651 -0.011445  0.009188 -0.011445 -0.002926   
2015-04-02 -0.003536 -0.000194 -0.000292  0.009176 -0.000292 -0.018254   
2015-04-06  0.003598 -0.008692  0.005700  0.009220  0.005700 -0.010208   
2015-04-07  0.006733  0.006782  0.010159  0.008797  0.010159 -0.012262   

              mom_20       rsi      volz  
2015-03-31 -0.013257  0.599115 -0.761779  
2015-04-01 -0.017850  0.563271 -0.042268  
2015-04-02 -0.017180  0.488343  0.220730  
2015-04-06 -0.014722  0.538022 -0.953142  
2015-04-07  0.006061  0.504730 -0.278628  
X shape: (2345, 9)  | y value counts:
 bin
 1    1394
-1     951
Name: count, dtype: int64


In [9]:
# 8) Train baseline classifier with time-aware CV (forward-chaining) and diagnostics

def train_evaluate_baseline_classifier(X: pd.DataFrame, y: pd.Series, t1: pd.Series,
                                       model_type: str = "logistic_regression",
                                       standardize: bool = True,
                                       purged_kfold_splits: int = 5,
                                       embargo_frac: float = 0.01,
                                       random_state: int = 42):
    # Sort by time to ensure forward chaining
    X = X.copy()
    y = y.loc[X.index]
    t1 = t1.loc[X.index]
    X.sort_index(inplace=True)
    y = y.loc[X.index]
    t1 = t1.loc[X.index]

    n = len(X)
    fold_sizes = np.linspace(0.6, 0.9, purged_kfold_splits)  # increasing train size
    indices = np.arange(n)

    if model_type == "logistic_regression":
        base_clf = LogisticRegression(max_iter=200, random_state=random_state, n_jobs=None)
    else:
        raise ValueError("Unsupported model_type")

    if standardize:
        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", base_clf)
        ])
    else:
        clf = base_clf

    oof_proba = pd.Series(index=X.index, dtype=float)
    oof_pred = pd.Series(index=X.index, dtype=int)
    reports = []

    times = X.index
    for i, frac in enumerate(fold_sizes, start=1):
        train_end = int(n * frac)
        test_end = int(n * min(frac + (1.0 - fold_sizes[0])/(purged_kfold_splits), 1.0))
        if test_end <= train_end or train_end <= 10:
            continue
        # Embargo: drop a small window after train_end from test start
        embargo = int((test_end - train_end) * embargo_frac)
        test_start = train_end + embargo
        if test_start >= test_end:
            continue
        tr_idx = indices[:train_end]
        te_idx = indices[test_start:test_end]

        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        pred = (proba >= 0.5).astype(int) if set(y.unique())=={0,1} else clf.predict(X_te)

        oof_proba.iloc[te_idx] = proba
        oof_pred.iloc[te_idx] = pred

        fold_report = {
            "fold": i,
            "train_range": (times[tr_idx[0]], times[tr_idx[-1]]),
            "test_range": (times[te_idx[0]], times[te_idx[-1]]),
            "auc": roc_auc_score(y_te, proba) if len(np.unique(y_te))>1 else np.nan,
            "accuracy": accuracy_score(y_te, pred),
            "report": classification_report(y_te, pred, output_dict=True),
            "confusion_matrix": confusion_matrix(y_te, pred).tolist(),
        }
        reports.append(fold_report)
        print(f"Fold {i}: AUC={fold_report['auc']:.3f}, ACC={fold_report['accuracy']:.3f}")

    cv_report = {"folds": reports}
    return oof_proba, oof_pred, cv_report

# Train/evaluate
oof_proba, oof_pred, cv_report = train_evaluate_baseline_classifier(
    X, y, t1_used,
    model_type="logistic_regression",
    standardize=True,
    purged_kfold_splits=5,
    embargo_frac=0.01,
    random_state=42,
)

print("Total coverage of OOF preds:", oof_pred.notna().mean())
# Quick summary
valid_mask = oof_pred.notna()
print("OOF AUC:", roc_auc_score(y[valid_mask], oof_proba[valid_mask]) if len(np.unique(y[valid_mask]))>1 else np.nan)
print("OOF ACC:", accuracy_score(y[valid_mask], oof_pred[valid_mask]))

Fold 1: AUC=0.524, ACC=0.554
Fold 2: AUC=0.410, ACC=0.487
Fold 3: AUC=0.452, ACC=0.583
Fold 4: AUC=0.579, ACC=0.631
Fold 5: AUC=0.483, ACC=0.636
Total coverage of OOF preds: 0.3795309168443497
OOF AUC: 0.464568435530323
OOF ACC: 0.5752808988764045


/Users/lakshya/miniconda3/envs/finagentv2/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lakshya/miniconda3/envs/finagentv2/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/lakshya/miniconda3/envs/finagentv2/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 

In [10]:
# 9) Build event-driven strategy weights for SPY

def build_event_strategy_weights(event_index: pd.DatetimeIndex,
                                 t1: pd.Series,
                                 oof_proba: pd.Series,
                                 proba_threshold: float = 0.5,
                                 holding_rule: str = "hold_until_t1",
                                 overlap_aggregation: str = "average_signal") -> pd.DataFrame:
    # Binary long-only direction: proba >= threshold => long, else flat
    signal = (oof_proba.loc[event_index] >= proba_threshold).astype(float)
    # Build daily signal timeline holding until t1
    e_idx = pd.DatetimeIndex(event_index)
    t1_idx = pd.DatetimeIndex(pd.to_datetime(t1.dropna()))
    combo = np.unique(np.concatenate([e_idx.values, t1_idx.values]))
    timeline_index = pd.DatetimeIndex(combo).sort_values()

    timeline = pd.DataFrame(index=timeline_index)
    timeline["signal_sum"] = 0.0
    timeline["signal_count"] = 0.0

    for t0 in e_idx:
        t_end = t1.loc[t0]
        if pd.isna(t_end):
            continue
        s = signal.loc[t0]
        # Slice [t0, t_end]
        idx = timeline.loc[t0:t_end].index
        timeline.loc[idx, "signal_sum"] += s
        timeline.loc[idx, "signal_count"] += 1.0

    if overlap_aggregation == "average_signal":
        agg = timeline["signal_sum"] / timeline["signal_count"].replace(0, np.nan)
    else:
        agg = (timeline["signal_sum"] > 0).astype(float)

    weights = pd.DataFrame(index=timeline.index, data={"SPY": agg.fillna(0.0)})
    return weights

asset_weights = build_event_strategy_weights(
    event_index=event_index,
    t1=t1_used,
    oof_proba=oof_proba,
    proba_threshold=0.5,
    holding_rule="hold_until_t1",
    overlap_aggregation="average_signal",
)

print(asset_weights.head(10))
print(asset_weights.tail(10))
print("Weights summary:\n", asset_weights.describe())

            SPY
2015-03-31  0.0
2015-04-01  0.0
2015-04-02  0.0
2015-04-06  0.0
2015-04-07  0.0
2015-04-08  0.0
2015-04-09  0.0
2015-04-10  0.0
2015-04-13  0.0
2015-04-14  0.0
            SPY
2024-12-17  0.0
2024-12-18  0.0
2024-12-19  0.0
2024-12-20  0.0
2024-12-23  0.0
2024-12-24  0.0
2024-12-26  0.0
2024-12-27  0.0
2024-12-30  0.0
2024-12-31  0.0
Weights summary:
                SPY
count  2350.000000
mean      0.378527
std       0.483073
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [11]:
# 10) Compute portfolio returns from weights and SPY prices

def compute_portfolio_returns_from_weights(weights: pd.DataFrame, prices: pd.DataFrame, price_field: str = "Close"):
    # Align to price index
    px = prices[price_field].copy().reindex(weights.index).ffill().dropna()
    # Daily returns
    ret = px.pct_change().fillna(0.0)
    # Use previous day’s weight (avoid lookahead)
    w = weights.reindex(ret.index).fillna(0.0).shift(1).fillna(0.0)
    # Single asset SPY
    strat_ret = (w["SPY"] * ret).to_frame(name="SPY")
    return strat_ret

asset_returns = compute_portfolio_returns_from_weights(asset_weights, ohlcv_df, price_field="Close")

print(asset_returns.head())
print(asset_returns.describe())

            SPY
2015-03-31  0.0
2015-04-01 -0.0
2015-04-02  0.0
2015-04-06  0.0
2015-04-07 -0.0
               SPY
count  2350.000000
mean      0.000188
std       0.006426
min      -0.043482
25%       0.000000
50%       0.000000
75%      -0.000000
max       0.054954


In [12]:
# 11) Final backtest call — must be called exactly once
# Assumes a user-defined function `backtest(weights: pd.DataFrame, prices: pd.Series|pd.DataFrame) -> pd.DataFrame`
# We pass SPY close prices to the backtest alongside the daily weights.

try:
    asset_returns = backtest(asset_weights, ohlcv_df["Close"])  # user-defined backtest
except NameError:
    # Fallback: define a minimal backtest compatible with the expected signature
    def backtest(weights: pd.DataFrame, prices):
        px = prices.copy()
        if isinstance(px, pd.DataFrame):
            # Expect single column Close for SPY
            px = px["Close"] if "Close" in px.columns else px.squeeze()
        px = px.reindex(weights.index).ffill()
        ret = px.pct_change().fillna(0.0)
        w = weights.reindex(ret.index).fillna(0.0).shift(1).fillna(0.0)
        return (w["SPY"] * ret).to_frame(name="SPY")
    asset_returns = backtest(asset_weights, ohlcv_df["Close"])  # exactly once

# Final artifacts
print("Final objects ready: asset_weights (DataFrame) and asset_returns (DataFrame)")
print(asset_weights.tail())
print(asset_returns.tail())

Final objects ready: asset_weights (DataFrame) and asset_returns (DataFrame)
            SPY
2024-12-24  0.0
2024-12-26  0.0
2024-12-27  0.0
2024-12-30  0.0
2024-12-31  0.0
            SPY
2024-12-24  0.0
2024-12-26  0.0
2024-12-27 -0.0
2024-12-30 -0.0
2024-12-31 -0.0
